Joud Alrubaish  - CourseFinder: Course Recommendation Assistant Using RAG

In [3]:
#Importing the required libraries 
import pandas as pd 
import numpy as np 
from sentence_transformers import SentenceTransformer
import faiss
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv
import os

/Users/joudalrubaish/capston_1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Phase 1 — Choose Your Domain & Collect Data
Since the time is limited , I picked Coursera Platform Course Recommendationomain System as a domain using Kaggle Coursera Course Dataset in csv format. 
This is data scrapped from Coursera website.

In [12]:
#phase 1 - loading coursera dataset 
raw_data = "data/raw/Coursea_data.csv"  
df = pd.read_csv(raw_data)

#checking 
print(df.shape)
print(df.columns)
df.head()

(891, 7)
Index(['Unnamed: 0', 'course_title', 'course_organization',
       'course_Certificate_type', 'course_rating', 'course_difficulty',
       'course_students_enrolled'],
      dtype='str')


,Unnamed: 0,course_title,course_organization,course_Certificate_type,course_rating,course_difficulty,course_students_enrolled
0,134,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k
1,743,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k
2,874,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k
3,413,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k
4,635,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k


# Phase 2 — Data Preprocessing & Chunking
This phase devided into two steps. I Cleaned the raw text by removin irrelavant column, duplicate values,and filling the missing values. Then, I created the documents to be splitted into chunks of tockens. 

# Justify your chunking strategy briefly: In this dataset each row represents one complete course. So instead of splitting the text into fixed-size chunks (300–800 tokens), each course record was treated as a single chunk.
because splitting a row would separate related attributes (title, organization, rating, difficulty, and enrollment), and each document contains fewer than 100 tokens, so additional chunking is unnecessary.

In [ ]:
#phase 2 - step 1(Data Preprocessing)
# Remove unnamed column
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Removing duplicate rows
df = df.drop_duplicates()

# Filling the missing values 
df["course_organization"] = df["course_organization"].fillna("Organization not specified")

df["course_Certificate_type"] = df["course_Certificate_type"].fillna("Certificate type not specified")

df["course_difficulty"] = df["course_difficulty"].fillna("Difficulty not specified")

df["course_students_enrolled"] = df["course_students_enrolled"].fillna("Enrollment data not available")

df["course_rating"] = df["course_rating"].fillna("Rating not available")

# Cleaning text columns
text_columns = [
    "course_title",
    "course_organization",
    "course_Certificate_type",
    "course_difficulty",
    "course_students_enrolled"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# Convert rating to string for before document creation
df["course_rating"] = df["course_rating"].astype(str)

#checking
print(df.shape)
df.head()

(891, 6)


,course_title,course_organization,course_Certificate_type,course_rating,course_difficulty,course_students_enrolled
0,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k
1,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k
2,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k
3,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k
4,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k


In [ ]:
#phase 2 - step 2(Data Chunking)
# converting each row into document 
documents = []

for index, row in df.iterrows():
    text = f"""
    Course Title: {row['course_title']}
    Organization: {row['course_organization']}
    Certificate Type: {row['course_Certificate_type']}
    Rating: {row['course_rating']}
    Difficulty Level: {row['course_difficulty']}
    Students Enrolled: {row['course_students_enrolled']}
    """
    
    documents.append(text.strip())

chunks = documents

#checking
print("Number of documents:", len(documents))
print("Number of chunks:", len(chunks))
print(chunks[0])

Number of documents: 891
Number of chunks: 891
Course Title: (ISC)² Systems Security Certified Practitioner (SSCP)
    Organization: (ISC)²
    Certificate Type: SPECIALIZATION
    Rating: 4.7
    Difficulty Level: Beginner
    Students Enrolled: 5.3k


# Phase 3 — Embeddings & Vector Database
This phase devided into three steps.I choose SentenceTransformer("all-MiniLM-L6-v2")for the embedding model to generate embeddings for all chunks.
- lightweight and computationally efficient.
- widely used in RAG applications.
- captures semantic similarity rather than simple matching.

 Then, store the embeddings in a vector database FAISS (fast search , common with RAG ) and build a retrieval function that takes a query, embeds it, using the same SentenceTransformer model, then searches the FAISS vector database to return the top-k most relevant chunks.

In [28]:
#phase 3 - step 1(Embedding Model)
# Loading SentenceTransformer embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generating embeddings vectors for the generated documents
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 28/28 [00:00<00:00, 40.67it/s]

Embedding shape: (891, 384)


In [30]:
#phase 3 - step 2 (FAISS)
#store the embeddings in a vector database FAISS

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total indexed documents:", index.ntotal)

Total indexed documents: 891


In [42]:
#phase 3 - step 3 (Retrivalfunction)
#build a retrieval function that takes a query, embeds it, using the same SentenceTransformer model
def retrieve_chunks(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
#searches the FAISS vector to return the top-k most relevant chunks
    distances, indices = index.search(query_embedding, top_k)

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return retrieved_chunks

In [44]:
#checking
#test_query = "Recommend machine learning courses"
#test_query = "Suggest business courses with high ratings"
#test_query = "Show AI courses"

ttest_query = "Suggest business courses with high ratings"

retrieved_results = retrieve_chunks(test_query)

for chunk in retrieved_results:
    print(chunk)
    print("-" * 60)

Course Title: Marketing Strategy
    Organization: IE Business School
    Certificate Type: SPECIALIZATION
    Rating: 4.4
    Difficulty Level: Beginner
    Students Enrolled: 57k
------------------------------------------------------------
Course Title: Foundations of Business Strategy
    Organization: University of Virginia
    Certificate Type: COURSE
    Rating: 4.8
    Difficulty Level: Beginner
    Students Enrolled: 63k
------------------------------------------------------------
Course Title: Strategic Business Management - Microeconomics
    Organization: University of California, Irvine
    Certificate Type: COURSE
    Rating: 4.7
    Difficulty Level: Beginner
    Students Enrolled: 9.5k
------------------------------------------------------------
Course Title: Business Strategy
    Organization: University of Virginia
    Certificate Type: SPECIALIZATION
    Rating: 4.7
    Difficulty Level: Beginner
    Students Enrolled: 89k
---------------------------------------------

# Phase 4 — Generation (LLM Integration)
This phase devided into three steps. I Choose openrouter (nvidia/nemotron-3-super-120b-a12b:free) LLM to generate the answers. I designed the prompt template that combines the user's question with the retrieved chunks.
I also ensured that the model answers only from the retrieved context, and clearly states when the answer is not found in the data.

In [56]:
# Phase 4 — step 1 (create openai key)

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [57]:
# Phase 4 — step 2 (prompt template)
def build_prompt(question, retrieved_chunks):

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are an intelligent assistant for Coursera Courses Recommendation.

Answer the user's question ONLY using the retrieved course information.

If the answer cannot be found in the provided context, respond exactly with:

"I could not find enough information in the dataset."

Retrieved Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [58]:
# Phase 4 — step 3 (generate answers)
def generate_answers(question):

    retrieved = retrieve_chunks(question)

    prompt = build_prompt(question, retrieved)

    response = client.chat.completions.create(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

# Phase 5 — Evaluation
I prepared a small set of test questions relevant to Coursera course dataset.The evaluation focused on: answer relevance, correctness/faithfulness to the retrieved context, and retrieval quality (are the right chunks retrieved?).

# Report your findings briefly (what worked well, what didn't, and why):
- The system works well in retrieving relevant course information by keeping the generated responses grounded in the retrieved context and did not introduce unsupported information.
- The main limitation is that the system can only answer questions related to the available Coursera course dataset.

In [59]:
# Phase 5 — Evaluation questions set 
evaluation_questions = [
    "Find Python courses.",
    "Recommend business courses.",
    "Which courses are offered by Google?",
    "Recommend beginner data science courses.",
    "What is RAG?"
]

for question in evaluation_questions:
    print("="*40)
    print("Question:", question)
    print()
    print(generate_answers(question))
    print()

Question: Find Python courses.

Here are the Python courses available in the dataset:

- **Aprende a programar con Python** – Universidad Austral – Specialization – Rating: 4.2 – Beginner – 6.6k students  
- **Introdução à Ciência da Computação com Python Parte 1** – Universidade de São Paulo – Course – Rating: 4.9 – Beginner – 120k students  
- **Python for Everybody** – University of Michigan – Specialization – Rating: 4.8 – Beginner – 1.5m students  
- **Python Functions, Files, and Dictionaries** – University of Michigan – Course – Rating: 4.8 – Beginner – 26k students  
- **Introducción a la programación en Python I: Aprendiendo a programar con Python** – Pontificia Universidad Católica de Chile – Course – Rating: 4.4 – Beginner – 100k students

Question: Recommend business courses.

Here are some business courses from the dataset:

- **Marketing Strategy** – IE Business School, Specialization, Rating 4.4, Beginner, 57k students enrolled  
- **Business Technology Management** – In

Phase 6 — Build the Gradio Demo
17.Build a simple, clean Gradio interface where a user can type a question and receive an answer.
18.Display the retrieved source chunks/documents alongside the generated answer (for transparency).
19.Add basic UI elements: input box, submit button, output area, and (optionally) example questions